In [ ]:
import requests
import json
import base64
import pandas as pd
import os

# url = "https://hiskenya.dha.go.ke/"

url = "https://histracker.dha.go.ke/"

KHIS_PASSWORD = os.getenv("KHIS_PASSWORD")

username = "Bmugwe"
password = KHIS_PASSWORD

credentials = f"{username}:{password}" 
auth_coded = base64.b64encode(credentials.encode()).decode("utf-8")

url = f"{url}api/users?query=&fields=username,displayName,id,organisationUnits[id,name],dataViewOrganisationUnits[id,name]&paging=false"
def fetch_data(url, auth_coded):
  payload = {}  
  headers = {
    'Authorization': f'Basic {auth_coded}',
    'Cookie': 'JSESSIONID=16D0523E692671A782DF2CE993E029E2'
  }
  response = requests.request("GET", url, headers=headers, data=payload)
  return response.json()

data = fetch_data(url, auth_coded)
user_data = data.get('users', [])


In [ ]:
data

In [ ]:
user_data_backup = pd.DataFrame(user_data)
user_data_backup.to_csv(f"{url.split('//')[1].split('/')[0]}_user_data_backup.csv", index=False)


In [ ]:
url = "https://hiskenya.dha.go.ke/api/29/dataSets/FqR4Q2B2VrY.html?fields=organisationUnits[id,name,code,level,parent[name,parent[name,parent[name]]]"

fetch_dataset_orgunits = fetch_data(url, auth_coded)


orgs = fetch_dataset_orgunits.get('organisationUnits', [])

In [ ]:
# org_df = pd.DataFrame(orgs)
orgs

In [ ]:
# orgs for parent field get the recursion and have each level of parent orgunit in a separate column.

def extract_parent_orgunits(orgunit):
    orgunit_data = {
        'id': orgunit.get('id'),
        'name': orgunit.get('name'),
        'code': orgunit.get('code'),
        'level': orgunit.get('level')
    }
    
    parent = orgunit.get('parent')
    level = 1
    
    while parent:
        orgunit_data[f'parent_level_{level}'] = parent.get('name')
        parent = parent.get('parent')
        level += 1
    
    return orgunit_data
    
orgs_with_parents = [extract_parent_orgunits(org) for org in orgs]
org_df = pd.DataFrame(orgs_with_parents)
org_df.to_csv("dataset_orgunits_with_parents.csv", index=False)

In [ ]:
def update_user(user_id):

  url = "https://hiskenya.dha.go.ke/api/40/users/{}".format(user_id)
  print("Url {}".format(url))

  payload = json.dumps(
[
    {
        "op": "add",
        "path": "/dataViewOrganisationUnits",
        "value": []
    },
    {
        "op": "add",
        "path": "/attributeValues",
        "value": []
    }
]
    )

  headers['Accept'] = 'application/json'
  headers['Content-Type'] = 'application/json-patch+json'
  

  response = requests.request("PATCH", url, headers=headers, data=payload)

  print(response.json())


In [ ]:
update_user("A0Mg7IMf51P")

# import concurrent.futures and run update_user in parallel for all users in batches of 20 to speed up the process.
import concurrent.futures

# print the number of users to be updated and save to log file
print("Number of users to be updated: {}".format(len(user_data)))

with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    executor.map(update_user, [user.get('id') for user in user_data if user.get('username') is not None])
    # 